In [1]:
import argparse

from collections import deque, defaultdict
import random
import math
import copy
import time
from typing import Tuple, List, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import csv
import matplotlib.pyplot as plt

import shutil
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFont

from some_utils import SokobanCurriculumEnvironment, BoardEncoder

In [ ]:
#Load the agent
from architecture import RotaryEmbedding, MLP, CausalAttentionBlock, CausalQueryAttention

class Config:
    n_embd = 64
    n_head = 4
    n_layers = 2
    block_size = 500
    dropout = 0.0
    bias = False
    conv_layers = [[32, 3], [64, 3]]
    grid_size = (6,6)
    initial_max_steps = 2
    replay_pool_capacity = 500
    replay_sample_prob = 0.2
    


class ActorCritic(nn.Module):
    def __init__(self, config:Config):
        super().__init__()
        self.config = config
        self.encoder = BoardEncoder(self.config)
        self.layers = nn.ModuleList()
        for _ in range(self.config.n_layers):
            self.layers.append(CausalAttentionBlock(self.config))

        self.policy = nn.Sequential(nn.Linear(self.config.n_embd, self.config.n_embd*2), nn.ReLU(), nn.Linear(self.config.n_embd*2, 4))
        self.value = nn.Sequential(nn.Linear(self.config.n_embd, self.config.n_embd*2), nn.ReLU(), nn.Linear(self.config.n_embd*2, 1))

    def forward(self, x, kv_cache=None):
        if x.dim() == 3:
            x = x.unsqueeze(0)
        
        h = self.encoder(x)
        for idx, layer in enumerate(self.layers):
            h, cache = layer(h, layer_past=kv_cache[idx])
            kv_cache[idx] = cache

        return self.policy(h), self.value(h), kv_cache
    
    def fast_causal_forward(self, B:torch.Tensor, f):
        """
        Takes advantage of compounded plans
        """
        T = self.encoder(B) # (B, Ch, H, W) -> (B, T, d)
        i  =  1
        kv_cache = None
        outputs = []
        while i<T.size(1):
            k = f(i)
            if i == 1:
                kv_cache = []
                for layer in self.layers:
                    out, cache = layer(T[:,i-1:i-1+k,:], k=None)
                    kv_cache.append(cache)
                    outputs.append(out)
            else:
                for idx, layer in enumerate(self.layers):
                    out, cache = layer(T[:,i-1:i-1+k,:], k=kv_cache[idx])
                    kv_cache[idx] = cache
                    outputs.append(out)
            i += k
        

agent = ActorCritic(Config())
env_helper = SokobanCurriculumEnvironment(Config())


In [26]:
class Loose_curr:
    push_pos = (0,0)

def sample_goal_box(env):
    dirs = [(1,0),(-1,0),(0,1),(0,-1)]

    while True:
        d = random.choice(dirs)
        gy = random.randint(0, 5)
        gx = random.randint(0, 5)

        by, bx = gy - d[0], gx - d[1]
        py, px = by - d[0], bx - d[1]

        G = (gy, gx)
        B = (by, bx)
        P = (py, px)

        if env.in_bounds(B) and env.in_bounds(P):
            return G, B, P
        
def sample_agent_path(env, P, B, N):
    visited = {P}
    path = [P]
    for _ in range(N):
        y, x = path[-1]
        candidates = []
        for dy, dx in env.action_map:
            ny, nx = y + dy, x + dx
            np = (ny, nx)
            if not env.in_bounds(np):
                continue
            if np in visited:
                continue
            if np == B:
                continue
            candidates.append(np)
        if not candidates:
            return None
        nxt = random.choice(candidates)
        visited.add(nxt)
        path.append(nxt)
    return path[::-1]

def sample_phase_N(env:SokobanCurriculumEnvironment, N):
    while True:
        G, B, P = sample_goal_box(env)
        path = sample_agent_path(env, P, B, N)
        if path is None:
            continue
        
        A = path[0]
        
        return A, B, G, P

def reset_zero(env:SokobanCurriculumEnvironment, use_replay:bool, stage:int, curriculum_obj:Loose_curr):
    # possibly sample from replay pool
    steps = [2, 4, 8, 12]
    if use_replay and len(env.replay_pool) > 0 and random.random() < env.replay_sample_prob:
        a, b, g, p = env.replay_pool.popleft()
    else:
        a, b, g, p = sample_phase_N(env, stage)
    env.load((a, set((b,)), set((g,)), set(())))
    curriculum_obj.push_pos = p
    env.max_steps = steps[stage]
    env.left_steps = env.max_steps
    env.finished = False

    return env.render_state()



tensor([[[0.5000, 0.2500, 0.7500, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.2500, 0.7500, 0.5000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.7500, 0.5000, 0.2500, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.

TypeError: cannot unpack non-iterable NoneType object